# 01 - Introduction to Vector Autoregression (VAR)

This notebook introduces **Vector Autoregressive (VAR)** models using the `chronobox` library.

## Topics covered

1. What is a VAR model?
2. Loading and exploring macroeconomic data
3. Lag order selection (AIC, BIC, HQIC, FPE)
4. Fitting a VAR model with `chronobox`
5. Interpreting coefficient matrices
6. Stability analysis (characteristic polynomial roots)
7. Forecasting
8. Exercises

---

### The VAR(p) model

A VAR(p) model for a $K$-dimensional time series $Y_t$ is:

$$Y_t = A_1 Y_{t-1} + A_2 Y_{t-2} + \cdots + A_p Y_{t-p} + c + u_t$$

where:
- $A_i$ are $K \times K$ coefficient matrices
- $c$ is a $K \times 1$ intercept vector
- $u_t \sim N(0, \Sigma_u)$ is the innovation process

VAR models capture **linear interdependencies** among multiple time series. Each variable
is modeled as a linear function of its own past values and the past values of all other
variables in the system.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
import os

# chronobox imports
from chronobox import VAR
from chronobox.selection.lag_selection import select_lag_order

# Plot helpers from the examples utilities
sys.path.insert(0, os.path.join("..", "utils"))
from plot_helpers import plot_multivariate_series

%matplotlib inline
plt.rcParams["figure.dpi"] = 100
np.set_printoptions(precision=4, suppress=True)

print("All imports loaded successfully.")

## 1. Loading the Canadian macroeconomic dataset

We use the classic **Canadian macroeconomic dataset** (Lutkepohl, 2005), which contains
quarterly observations from 1980Q1 to 2000Q4 for four variables:

| Variable | Description |
|----------|-------------|
| `e`      | Employment rate |
| `prod`   | Labour productivity |
| `rw`     | Real wage |
| `U`      | Unemployment rate |

In [ ]:
# Load data
data_path = os.path.join("..", "data", "canada_macro.csv")
df = pd.read_csv(data_path)
print(f"Shape: {df.shape}")
print(f"Period: {df['date'].iloc[0]} to {df['date'].iloc[-1]}")
df.head()

In [ ]:
# Visualize the four series
fig = plot_multivariate_series(df, date_col="date", title="Canadian Macro Variables (1980-2000)")
plt.savefig(os.path.join("..", "outputs", "canada_series.png"), bbox_inches="tight")
plt.show()

**Economic interpretation:** The four series show clear trends and co-movement.
Productivity (`prod`) and real wages (`rw`) trend upward over time, while employment
(`e`) and unemployment (`U`) show business cycle fluctuations. This co-movement
suggests that a VAR model can capture the dynamic interactions between these variables.

## 2. Lag order selection

Choosing the right lag order $p$ is crucial. Too few lags miss important dynamics;
too many waste degrees of freedom and risk overfitting.

We use four information criteria:

| Criterion | Formula | Tendency |
|-----------|---------|----------|
| AIC  | $\ln|\hat\Sigma_u| + \frac{2pK^2}{T}$ | Tends to overfit |
| BIC  | $\ln|\hat\Sigma_u| + \frac{pK^2 \ln T}{T}$ | More parsimonious |
| HQIC | $\ln|\hat\Sigma_u| + \frac{2pK^2 \ln(\ln T)}{T}$ | Intermediate |
| FPE  | $(\frac{T+Kp+1}{T-Kp-1})^K |\hat\Sigma_u|$ | Finite prediction error |

In [ ]:
# Prepare the endogenous data (drop the date column)
endog = df[["e", "prod", "rw", "U"]].values
var_names = ["e", "prod", "rw", "U"]

# Lag order selection up to 8 lags
lag_results = select_lag_order(endog, maxlags=8, trend="c")
print(lag_results.summary())

In [ ]:
# Plot information criteria across lag orders
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle("Information Criteria by Lag Order", fontsize=14)

criteria = {"AIC": lag_results.aic, "BIC": lag_results.bic,
            "HQIC": lag_results.hqic, "FPE": lag_results.fpe}

for ax, (name, vals) in zip(axes.flat, criteria.items()):
    lags = sorted(vals.keys())
    values = [vals[l] for l in lags]
    ax.plot(lags, values, "o-", color="steelblue")
    best = min(vals, key=vals.get)
    ax.axvline(best, color="red", linestyle="--", alpha=0.7, label=f"Best: p={best}")
    ax.set_title(name)
    ax.set_xlabel("Lag order")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join("..", "outputs", "lag_selection_criteria.png"), bbox_inches="tight")
plt.show()

print("\nSelected lag orders:")
for crit, order in lag_results.selected_orders.items():
    print(f"  {crit}: p = {order}")

**Interpretation:** AIC often selects higher-order models because its penalty for
additional parameters is lighter. BIC is the most parsimonious criterion and typically
selects fewer lags. In practice, one should consider the trade-off: BIC is consistent
(selects the true order as $T \to \infty$) while AIC is efficient (minimizes forecast
MSE for finite samples).

## 3. Fitting the VAR model

We fit a VAR(2) model based on the information criteria results. The `chronobox.VAR`
class estimates the model via multivariate OLS.

In [ ]:
# Fit VAR(2)
model = VAR(lags=2, trend="c")
results = model.fit(endog, names=var_names)

print(results.summary())

In [ ]:
# Display coefficient matrices
print("Coefficient matrix A_1 (lag 1):")
A1 = pd.DataFrame(results.coefs[0], index=var_names, columns=var_names)
print(A1.round(4))

print("\nCoefficient matrix A_2 (lag 2):")
A2 = pd.DataFrame(results.coefs[1], index=var_names, columns=var_names)
print(A2.round(4))

print(f"\nIntercept: {results.intercept.round(4)}")

print(f"\nResidual covariance matrix:")
print(pd.DataFrame(results.sigma_u, index=var_names, columns=var_names).round(6))

**Economic interpretation of A_1:** The diagonal elements show the persistence of each
variable (how strongly its own first lag predicts the current value). Off-diagonal
elements capture cross-variable dynamics -- for example, the coefficient of lagged
unemployment in the employment equation captures how unemployment shocks propagate
to employment one quarter later.

## 4. Stability analysis

A VAR(p) process is **stable** (stationary) if all eigenvalues of the companion matrix
lie **inside the unit circle** ($|\lambda_i| < 1$). If any root has modulus $\geq 1$,
the process is non-stationary.

The companion matrix $F$ for a VAR(p) is:

$$F = \begin{pmatrix} A_1 & A_2 & \cdots & A_p \\ I_K & 0 & \cdots & 0 \\ 0 & I_K & \cdots & 0 \\ \vdots & & \ddots & \vdots \\ 0 & 0 & \cdots & 0 \end{pmatrix}$$

In [ ]:
# Check stability
print(f"Is the VAR stable? {results.is_stable}")
print(f"\nEigenvalues of the companion matrix:")
for i, root in enumerate(results.roots):
    print(f"  lambda_{i+1} = {root:.4f}  (modulus = {abs(root):.4f})")

# Plot eigenvalues on the unit circle
fig, ax = plt.subplots(figsize=(6, 6))
theta = np.linspace(0, 2 * np.pi, 200)
ax.plot(np.cos(theta), np.sin(theta), "k-", linewidth=0.8, label="Unit circle")

roots = results.roots
ax.scatter(roots.real, roots.imag, color="red", s=80, zorder=5, label="Eigenvalues")

ax.set_xlabel("Real")
ax.set_ylabel("Imaginary")
ax.set_title("VAR Stability: Eigenvalues and Unit Circle")
ax.set_aspect("equal")
ax.legend()
ax.grid(True, alpha=0.3)
plt.savefig(os.path.join("..", "outputs", "var_stability.png"), bbox_inches="tight")
plt.show()

print(f"\nMaximum modulus: {max(abs(roots)):.4f}")
if results.is_stable:
    print("All eigenvalues inside the unit circle => process is stable.")
else:
    print("WARNING: Some eigenvalues outside the unit circle => process is non-stationary!")

## 5. Model diagnostics

We check the whiteness of residuals using the Portmanteau test. Under the null hypothesis,
residuals are serially uncorrelated (white noise).

In [ ]:
# Whiteness test
whiteness = results.test_whiteness(nlags=10)
print(f"Portmanteau test statistic: {whiteness['statistic']:.4f}")
print(f"p-value: {whiteness['pvalue']:.4f}")
print(f"Degrees of freedom: {whiteness['df']}")

if whiteness['pvalue'] > 0.05:
    print("=> Cannot reject H0: residuals are white noise (good).")
else:
    print("=> Reject H0: residuals are NOT white noise (consider more lags).")

## 6. Forecasting

The VAR model can produce $h$-step-ahead forecasts. The point forecast for horizon $h$ is:

$$\hat{Y}_{T+h|T} = A_1 \hat{Y}_{T+h-1|T} + \cdots + A_p \hat{Y}_{T+h-p|T} + c$$

In [ ]:
# Forecast 8 steps ahead
fig = results.plot_forecast(steps=8, alpha=0.05)
plt.suptitle("VAR(2) Forecast: 8 Quarters Ahead", fontsize=14, y=1.02)
plt.savefig(os.path.join("..", "outputs", "var_forecast.png"), bbox_inches="tight")
plt.show()

In [ ]:
# Information criteria for the fitted model
print("Model fit statistics:")
print(f"  AIC:  {results.aic:.4f}")
print(f"  BIC:  {results.bic:.4f}")
print(f"  HQIC: {results.hqic:.4f}")
print(f"  FPE:  {results.fpe:.6f}")
print(f"  Observations (effective): {results.nobs}")
print(f"  Number of equations: {results.neqs}")
print(f"  Lag order: {results.k_ar}")

## Exercicio

### Exercicio 1: VAR com dados US Macro

Use o dataset `us_macro_quarterly.csv` (GDP growth, inflation, fed funds rate, unemployment)
para:

1. Carregar os dados e visualizar as series
2. Realizar selecao de ordem ate 12 lags
3. Ajustar um VAR com a ordem selecionada por BIC
4. Verificar estabilidade do modelo
5. Interpretar os coeficientes da equacao de inflacao

**Dica:** Use `select_lag_order(endog, maxlags=12)` e depois `VAR(lags=p_bic)`.

In [ ]:
# --- Exercicio 1: Seu codigo aqui ---

# 1. Carregar os dados US macro
us_path = os.path.join("..", "data", "us_macro_quarterly.csv")
us_df = pd.read_csv(us_path)
print(us_df.head())

# 2. Visualizar
fig = plot_multivariate_series(us_df, date_col="date", title="US Macro Variables")
plt.show()

# 3. Selecao de ordem
us_endog = us_df[["gdp", "inflation", "fed_funds", "unemployment"]].values
us_names = ["gdp", "inflation", "fed_funds", "unemployment"]

us_lag = select_lag_order(us_endog, maxlags=12, trend="c")
print(us_lag.summary())

# 4. Ajustar VAR com lag selecionado por BIC
p_bic = us_lag.selected_orders["bic"]
print(f"\nBIC selects p = {p_bic}")

us_model = VAR(lags=p_bic, trend="c")
us_results = us_model.fit(us_endog, names=us_names)
print(us_results.summary())

# 5. Estabilidade
print(f"\nStable? {us_results.is_stable}")
print(f"Max eigenvalue modulus: {max(abs(us_results.roots)):.4f}")

### Exercicio 2: Comparacao de ordens

Ajuste VAR(1), VAR(2), VAR(3) e VAR(4) nos dados canadenses e compare:
- AIC de cada modelo
- Estabilidade de cada modelo
- Resultado do teste de whiteness para cada modelo

Qual ordem voce escolheria e por que?

In [ ]:
# --- Exercicio 2: Seu codigo aqui ---

for p in [1, 2, 3, 4]:
    m = VAR(lags=p, trend="c")
    r = m.fit(endog, names=var_names)
    w = r.test_whiteness(nlags=10)
    print(f"VAR({p}): AIC={r.aic:.2f}, BIC={r.bic:.2f}, "
          f"Stable={r.is_stable}, Whiteness p-val={w['pvalue']:.4f}")

---

## Resumo

Neste notebook aprendemos:

1. **VAR(p)** modela interdependencias lineares entre multiplas series temporais
2. **Selecao de ordem** usa criterios de informacao (AIC, BIC, HQIC, FPE) para equilibrar ajuste e parcimonia
3. **Estabilidade** requer que todas as raizes do polinomio caracteristico estejam dentro do circulo unitario
4. **Diagnosticos** (teste de whiteness) verificam se os residuos sao ruido branco
5. **Previsao** e feita iterativamente usando as matrizes de coeficientes estimadas

No proximo notebook, veremos como lidar com series **cointegradas** usando o modelo VECM.